In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 

## Adaboost classifier implementation

In [2]:
from sklearn.tree import DecisionTreeClassifier

In [11]:
class MyAdaBoostClassifier: 
    def __init__(self , n_estimators = 50): 
        self.n_estimators = n_estimators # num of decision stumps we want to use as weak learner
        self.alphas = [] # weights of all weak learners
        self.models = [] # stores all the objects of weak learners

    def fit(self , X , y): 
        n_samples , _ = X.shape
        
        # convert labels to -1 and +1
        y = np.where(y == 0 , -1 , 1)

        # Initialize sample weights(each row weights will be 1/n where n is total rows) 
        w = np.ones(n_samples) / n_samples

        # now train each decision stumps
        for estimator in range(self.n_estimators): 
            # Train a weak leaner(decision stumps)
            stump = DecisionTreeClassifier(max_depth = 1)
            stump.fit(X , y , sample_weight = w) 
            # Do the predictions 
            y_pred = stump.predict(X)

            # Find the weight for the current estimator
            # 1. Find the error sum 
            error = np.sum(w * (y_pred != y)) / np.sum(w) 

            # avoid dividing by zero 
            if error == 0: # leaner confidence level is 100% so give it a very high weight 
                alpha = 1e10
            elif error == 1: # learner confidence is -100% so break the training phase
                raise RuntimeWarning("Found one learner with worst prediction than random")
                break
            else:
                alpha = 0.5 * np.log((1 - error) / error) 

            # update the weights
            w = w * np.exp(-alpha * y * y_pred)
            w /= np.sum(w) # normalizing weights

            # save the model and alpha 
            self.models.append(stump)
            self.alphas.append(alpha)

    def predict(self , X): 
        clf_preds = np.array([alpha * clf.predict(X) for clf , alpha in zip(self.models , self.alphas)])
        final_pred = np.sign(np.sum(clf_preds , axis = 0))
        return np.where(final_pred == -1 , 0 , 1) # back 0/1 from -1/+1

In [3]:
y = np.array([1,1,0,1,0,0])
y = np.where(y == 0 , -1 , 1)
y

array([ 1,  1, -1,  1, -1, -1])

In [5]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [6]:
# Create toy dataset
X, y = make_classification(n_samples=100, n_features=2, n_informative=2,
                           n_redundant=0, random_state=42)

In [7]:
# Convert to 0 and 1
y = np.where(y == 0, 0, 1)

In [8]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [12]:
# Train AdaBoost
clf = MyAdaBoostClassifier(n_estimators=10)
clf.fit(X_train, y_train)

In [13]:
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9666666666666667
